In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_file = Path.cwd() / "secrets.env"
load_dotenv(env_file)

groq_key = os.getenv("GROQ_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not groq_key:
    raise RuntimeError("GROQ_API_KEY is missing. Add it to secrets.env, then restart and run this cell.")

In [2]:
from openai import OpenAI

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_groq import ChatGroq

C:\Users\usama\AppData\Local\Temp\ipykernel_16840\3847616107.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
client = OpenAI(
    api_key= groq_key,
    base_url= 'https://api.groq.com/openai/v1/'
)

In [5]:
def check_emotion(prompt):
    response = client.chat.completions.create(
        model = 'openai/gpt-oss-20b',
        messages= [
            {
                'role' : 'system',
                'content': f"you'll be given a sentence , check it's emotion"
            },
            {
                'role' : 'user',
                'content': "I'm so lonely"
            },
            {
                'role' : 'assistant',
                'content' : 'sad'
            },
            {
                'role' : 'user',
                'content': prompt
            }
        ]
    )
    return response.choices[0].message.content.strip()


In [6]:
print(check_emotion('funny world we live in '))

amusement


In [7]:
text_loader = TextLoader(
    file_path= 'Games record .txt',
    encoding= 'utf-8'
)
raw_document = text_loader.load()

In [8]:
text_splitter = RecursiveCharacterTextSplitter()
splitted_docs = text_splitter.split_documents(raw_document)

In [9]:
embeddings = HuggingFaceEmbeddings(
     model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
vector_store = FAISS.from_documents(documents= splitted_docs, embedding= embeddings)

In [11]:
memory = ConversationBufferMemory(memory_key= "chat_history" , return_messages= True)

C:\Users\usama\AppData\Local\Temp\ipykernel_16840\2948440534.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key= "chat_history" , return_messages= True)


In [12]:
qa = ConversationalRetrievalChain.from_llm(
    ChatGroq(
        model= 'openai/gpt-oss-20b',
        api_key= groq_key,
        temperature= 0.7
    ),
    vector_store.as_retriever(),
    memory = memory
)

In [21]:
 query = 'any game with vin written alongside it ?'

In [22]:
print("qa.memory:", qa.memory)
print("memory key:", qa.memory.memory_key if qa.memory else None)
print(qa.prep_inputs({"question": "test"}))

qa.memory: chat_memory=InMemoryChatMessageHistory(messages=[HumanMessage(content='When was GTA 4 added?', additional_kwargs={}, response_metadata={}), AIMessage(content='GTA\u202f4 was added on **18‑4‑19**.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='is there any action games there ?', additional_kwargs={}, response_metadata={}), AIMessage(content='Yes – there are plenty of action‑oriented games in the list. Below is a selection of titles that fall into the **action, action‑adventure, first‑person shooter, or tactical‑action** categories (based on the titles’ typical gameplay styles).  \n\n| # | Game (release date) | Typical genre |\n|---|---------------------|---------------|\n| 1 | **Spec Ops: The Line** (2‑12‑2018) | Third‑person tactical shooter |\n| 2 | **Half‑Life\u202f2 Episode\u202f1** (13‑12‑18) | First‑person shooter |\n| 3 | **Left 4 Dead\u202f2** (9‑1‑19) | Co‑op first‑person shooter |\n| 4 | **Doom\u202f3** (9‑

In [23]:
result = qa.invoke({
    "question": query,
})

In [24]:
result['answer'].strip()

'Yes – the only entry marked with “vin” is:\n\n- **Borderlands\u202f2** (25‑4‑24)\u202f(vin)'